## 2. Formal problem statement

Problem version: Level A (Advanced), the Capacitated Vehicle Routing Problem with Time Windows (CVRPTW), with peak-hour traffic.

### 2.1 Problem definition

A fleet of at most $m$ identical vehicles of capacity $Q$ delivers goods from one depot to $n$ customers. Customer $i$ has a demand $q_i$, a service duration $d_i$ and a time window $[a_i,b_i]$ in which service must start. A vehicle that arrives early waits, and late service is forbidden. Travel times depend on the time slot in which the vehicle leaves, so a trip takes longer during the peak. We look for routes that start and end at the depot, serve every customer exactly once, respect the capacity and every time window, and have the smallest total cost. This is the vehicle routing problem of Dantzig and Ramser [1] with the time windows studied by Solomon [4]. Surveys of its formulations and exact methods are in [7, 8].

A solution is a set of routes, the start time $s_i$ of each service, and the total cost $Z$.

Assumptions:

* (A1) There is one depot. The vehicles are identical with capacity $Q$. At most $m$ are used, each on one route, and $m$ is a hard bound.
* (A2) Demands, service durations, time windows and travel times are known and deterministic, and $1\le q_i\le Q$.
* (A3) Each customer is served in one visit by one vehicle. Deliveries are not split.
* (A4) Time windows are hard. Waiting, before and after service, is allowed and costs nothing.
* (A5) Vehicles leave the depot at time $0$ and must be back by the horizon $H$. Leaving at $0$ loses nothing under the objective of 2.4 (see 2.7).
* (A6) The travel time of a trip is fixed by the slot in which the vehicle leaves. Speed changes during a trip are ignored.

### 2.2 Notation and traffic model

The problem lives on a directed graph $G=(V,A)$. The depot appears twice: node $0$ starts every route and node $n+1$ is a copy where every route ends. This lets the depot take part in the time constraints like any other node.

$$N=\{1,\dots,n\},\qquad V=\{0\}\cup N\cup\{n+1\},$$

$$A=\{(0,j):j\in N\}\ \cup\ \{(i,j):i,j\in N,\ i\ne j\}\ \cup\ \{(i,n+1):i\in N\}.$$

There is no arc into $0$, out of $n+1$, or from $0$ to $n+1$, so every route visits at least one customer.

| Symbol | Meaning |
|---|---|
| $n,\ m,\ Q$ | Number of customers, maximum number of vehicles, vehicle capacity |
| $q_i$ | Demand of customer $i$; $q_0=q_{n+1}=0$ |
| $a_i,\ b_i$ | Earliest and latest start of service at $i$. The depot window is $[0,H]$, so $a_{n+1}=0$ and $b_{n+1}=H$ |
| $d_i$ | Service duration at $i$, $d_0=d_{n+1}=0$. Setting $d_i=0$ ignores service times |
| $H$ | End of the planning horizon |
| $P,\ \theta_p$ | Number of traffic slots and their boundaries, $0=\theta_0<\dots<\theta_P=H$ |
| $t_{ij}$ | Free-flow travel time from $i$ to $j$ |
| $\rho_p$ | Congestion factor of slot $p$ ($1$ means free flow) |
| $t_{ij}^p$ | Travel time from $i$ to $j$ when leaving in slot $p$ |
| $c_{ij}$ | Cost of arc $(i,j)$, equal to $t_{ij}$ |

All times share one unit. On benchmark instances $t_{ij}$ is the Euclidean distance, rounded as the benchmark prescribes.

Slot $p$ is the closed interval $[\theta_{p-1},\theta_p]$ and $\mathcal{P}=\{1,\dots,P\}$. The default profile has $P=3$: off-peak on $[0,\theta_1]$, peak on $[\theta_1,\theta_2]$, off-peak on $[\theta_2,H]$, with $\rho_1=\rho_3=1$, $\rho_2=\rho\ge1$ (default $1.5$) and $t_{ij}^p=\rho_p\,t_{ij}$. Any other values of $t_{ij}^p$ can replace this simple peak. Piecewise constant travel times of this kind follow Malandraki and Daskin [11], and a review of time-dependent routing is in [13]. A departure exactly at a slot boundary may use either neighbouring slot, and the faster one is taken.

Setting $\rho=1$ gives the static CVRPTW. Setting in addition $a_i=0$, $b_i=H$ with $H$ large and $d_i=0$ gives the plain CVRP, which the vrplib instances require.

### 2.3 Decision variables

| Variable | Type | Meaning |
|---|---|---|
| $x_{ij}$, $(i,j)\in A$ | binary | 1 if a vehicle drives directly from $i$ to $j$ |
| $z_{ij}^p$, $(i,j)\in A$, $p\in\mathcal{P}$ | binary | 1 if arc $(i,j)$ is used and the vehicle leaves $i$ in slot $p$ |
| $s_i$, $i\in N\cup\{n+1\}$ | continuous | Start of service at $i$; for $i=n+1$ it is the return time |
| $\delta_i$, $i\in N$ | continuous | Time at which the vehicle leaves $i$; the departure from the depot is the constant $\delta_0=0$ |
| $u_i$, $i\in N$ | continuous | Cumulative demand delivered on the route up to and including $i$ |

No variable has a vehicle index. The vehicles are identical and each customer is visited once, so one $s_i$, one $\delta_i$ and one $u_i$ per customer are enough. Routes are read from $x$ by following the selected arcs from node $0$ to node $n+1$. Since $x_{ij}=\sum_p z_{ij}^p$, the variables $x$ are a sum of the variables $z$. We keep them because they make the degree and load constraints easier to read.

### 2.4 Objective function

$$\min\ Z=\sum_{(i,j)\in A}c_{ij}\,x_{ij},\qquad c_{ij}=t_{ij}.$$

$Z$ is the total free-flow travel time, which equals distance on the benchmarks. Waiting costs nothing, and the number of vehicles is bounded by $m$ but not penalised. Traffic therefore acts only through the time constraints: it can make a route infeasible, force a wait, or force a different order, while the cost of an arc stays the same. This keeps the results comparable with the static benchmarks and means an earlier schedule is never worse than a later one.

### 2.5 Complete ILP

$$\min\ \sum_{(i,j)\in A} c_{ij}\,x_{ij}$$

subject to

$$\sum_{j:(i,j)\in A} x_{ij}=1 \qquad \forall\, i\in N \tag{1}$$

$$\sum_{i:(i,j)\in A} x_{ij}=1 \qquad \forall\, j\in N \tag{2}$$

$$\sum_{j\in N} x_{0j}\ \le\ m \tag{3}$$

$$u_i-u_j+Q\,x_{ij}\ \le\ Q-q_j \qquad \forall\, i,j\in N,\ i\ne j \tag{4}$$

$$q_i\ \le\ u_i\ \le\ Q \qquad \forall\, i\in N \tag{5}$$

$$\sum_{p\in\mathcal{P}} z_{ij}^p = x_{ij} \qquad \forall\, (i,j)\in A \tag{6}$$

$$\delta_i\ \ge\ s_i+d_i \qquad \forall\, i\in N \tag{7}$$

$$\delta_i\ \ge\ \theta_{p-1}\,z_{ij}^p \qquad \forall\, (i,j)\in A,\ p\in\mathcal{P} \tag{8}$$

$$\delta_i\ \le\ \theta_p+(H-\theta_p)\,(1-z_{ij}^p) \qquad \forall\, (i,j)\in A,\ p\in\mathcal{P} \tag{9}$$

$$\delta_i+t_{ij}^p-s_j\ \le\ M_{ij}^p\,(1-z_{ij}^p) \qquad \forall\, (i,j)\in A,\ p\in\mathcal{P} \tag{10}$$

$$a_i\ \le\ s_i\ \le\ b_i \qquad \forall\, i\in N\cup\{n+1\} \tag{11}$$

$$x_{ij},\,z_{ij}^p\in\{0,1\},\qquad 0\le\delta_i\le H \tag{12}$$

where $\delta_0=0$ and

$$M_{ij}^p=\max\{\,0,\ \hat H_i+t_{ij}^p-a_j\,\},\qquad \hat H_0=0,\quad \hat H_i=H\ \ (i\in N).$$

| Constraint | Meaning |
|---|---|
| (1), (2) | Every customer is left once and entered once, so it is visited exactly once. |
| (3) | At most $m$ vehicles leave the depot. |
| (4) | If $j$ directly follows $i$, then $u_j\ge u_i+q_j$, so the load accumulates along the route. When $x_{ij}=0$ the constraint always holds. It is the load-based form of the Miller, Tucker and Zemlin constraints [2]. |
| (5) | The load after serving $i$ is at least $q_i$ and at most $Q$. |
| (6) | A used arc has exactly one departure slot, and an unused arc has none. |
| (7) | A vehicle cannot leave a customer before finishing service, and it may wait longer. |
| (8), (9) | If $z_{ij}^p=1$, the departure time $\delta_i$ lies in $[\theta_{p-1},\theta_p]$. If $z_{ij}^p=0$ both hold for any $\delta_i\in[0,H]$. |
| (10) | If $z_{ij}^p=1$, then $s_j\ge\delta_i+t_{ij}^p$: service at $j$ starts after the drive of the chosen slot. Being an inequality, it allows waiting before service. |
| (11) | Service starts inside the window. For $i=n+1$ it forces the return by $H$. |
| (12) | Domains of the binary variables and of $\delta$. |

The model has $(n^2+n)(P+1)$ binary variables, about $3n$ continuous variables and $O(n^2P)$ constraints, and none of these counts depends on $m$. With the three-slot profile there are four times as many binaries as in the static model. Because $\delta_0=0$, constraint (8) forces $z_{0j}^p=0$ for $p\ge2$, so an implementation can drop those variables. Strictly the model is a mixed integer program, since $s$, $\delta$ and $u$ are continuous. If all data are integers, including the values $t_{ij}^p$, an earliest schedule has integer times, so these variables can be declared integer without changing the optimum, which gives a pure ILP.

### 2.6 Route evaluation rule

Given a route $0\to i_1\to\dots\to i_r\to n+1$, set $s_0=0$. For each arc $(i,j)$ let $r_i=s_i+d_i$ be the time the vehicle is ready to leave $i$. Then

$$\text{arr}_{ij}(r_i)=\min_{p:\ \max(r_i,\theta_{p-1})\le\theta_p}\Big\{\max(r_i,\theta_{p-1})+t_{ij}^p\Big\},\qquad s_j=\max\{a_j,\ \text{arr}_{ij}(r_i)\}.$$

The route is feasible if every $s_j\le b_j$, $s_{n+1}\le H$ and the load is at most $Q$. The rule tries each slot that has not ended, leaves as early as that slot allows, and keeps the earliest arrival. It costs $O(P)$ per arc. The heuristic uses it to evaluate routes, and the ILP is its exact counterpart.

### 2.7 Correctness of the model

Subtours. Constraints (4) and (5) forbid any cycle $C\subseteq N$ with $x_{ij}=1$ on all its arcs. Summing (4) over the arcs of $C$, the terms $u_i-u_j$ cancel around the cycle and we get $Q|C|\le Q|C|-\sum_{j\in C}q_j$. This is impossible because $q_j\ge1$.

Value of $M_{ij}^p$. When $z_{ij}^p=0$, the left side of (10) is at most $\hat H_i+t_{ij}^p-a_j$, because $\delta_i\le\hat H_i$ and $s_j\ge a_j$, and this bound is reached. So $M_{ij}^p$ is the smallest constant that keeps (10) valid. We compute it for each arc and slot, since a single global $M$ weakens the LP relaxation and can cause numerical trouble. Because vehicles may wait, $\delta_i$ can be as large as $H$, which makes $M_{ij}^p$ larger than in the static model. This is the main reason the traffic model is harder for a solver.

Equivalence with the evaluation rule. A vector satisfying (1) to (12) has its arcs with $x_{ij}=1$ forming at most $m$ routes that partition $N$, and each route is feasible under the rule of 2.6. Along a path, (4) and (5) bound the load by $Q$. For a used arc, (6) gives one slot $p$, (8) and (9) put $\delta_i$ in that slot, (7) gives $\delta_i\ge s_i+d_i$, and (10) gives $s_j\ge\delta_i+t_{ij}^p$. Together with (11) this is a schedule that respects the windows and the horizon, and the rule gives start times no later than any such schedule.

Conversely, take a route that is feasible under the rule. For each arc choose the slot $p$ that attains the minimum, and set $z_{ij}^p=1$, $\delta_i=\max(r_i,\theta_{p-1})$, $s_j$ as in the rule, and $u_i$ equal to the cumulative load. All constraints hold on used arcs by construction. On unused arcs (8), (9) and (10) hold because $z=0$ and by the choice of $M_{ij}^p$, and (4) reduces to $u_i-u_j\le Q-q_j$, which follows from $u_i\le Q$ and $u_j\ge q_j$.

The rule gives the earliest start times because $r\mapsto\text{arr}_{ij}(r)$ is nondecreasing. Each term of the minimum is nondecreasing in $r$, and the set of admissible slots can only shrink as $r$ grows. So a vehicle that is ready earlier never arrives later, which also justifies the departure at time $0$ in (A5).

### 2.8 Modelling choice

We use a two-index formulation. The vehicles are identical, so a vehicle index $k$ adds nothing. A three-index model $x_{ijk}$ would need about $n^2m$ binaries (times $P$ with traffic) and would contain up to $m!$ equivalent copies of each solution, which slows branch and bound. The load constraints (4) are compact but give a weak LP bound. The set partitioning formulation, with one binary per feasible route and solved by column generation, has a much stronger bound and underlies the best exact methods [5, 8]. With traffic, route feasibility would be checked by the rule of 2.6 when a column is generated. We cite it for bounds and do not implement it.

### 2.9 Uses of the exact model

1. Direct solving of small instances, where a MILP solver returns a provably optimal solution. The reachable size depends on how tight the windows are, and it is smaller than for the static model because traffic multiplies the binaries by about $P+1$.
2. Reference for the heuristic. On every instance the solver closes, the optimal cost $C^*$ gives the exact gap $100\,(C_{\text{heur}}-C^*)/C^*$.
3. Lower bound. The LP relaxation gives $Z_{LP}\le C^*$. It is cheap but weak because of the big-M constraints. When the solver stops at a time limit, its dual bound is also a lower bound.
4. Upper bound. A solver stopped early returns a feasible solution, and a heuristic solution can be passed to it as a warm start.
5. Test oracle. Constraints (1) to (12) are an executable definition of feasibility, so the route evaluation and the moves of the heuristic can be checked against them on small instances.

### References cited in Section 2

1. Dantzig, G. B. and Ramser, J. H. (1959). The truck dispatching problem. *Management Science*, 6(1), 80-91.
2. Miller, C. E., Tucker, A. W. and Zemlin, R. A. (1960). Integer programming formulation of traveling salesman problems. *Journal of the ACM*, 7(4), 326-329.
4. Solomon, M. M. (1987). Algorithms for the vehicle routing and scheduling problems with time window constraints. *Operations Research*, 35(2), 254-265.
5. Desrochers, M., Desrosiers, J. and Solomon, M. (1992). A new optimization algorithm for the vehicle routing problem with time windows. *Operations Research*, 40(2), 342-354.
7. Cordeau, J.-F., Desaulniers, G., Desrosiers, J., Solomon, M. M. and Soumis, F. (2002). VRP with time windows. In Toth, P. and Vigo, D. (eds), *The Vehicle Routing Problem*, SIAM, 157-193. https://doi.org/10.1137/1.9780898718515.ch7 (accessed 18 October 2026).
8. Kallehauge, B., Larsen, J., Madsen, O. B. G. and Solomon, M. M. (2005). Vehicle routing problem with time windows. In Desaulniers, G., Desrosiers, J. and Solomon, M. M. (eds), *Column Generation*, Springer, 67-98. https://doi.org/10.1007/0-387-25486-2_3 (accessed 18 October 2026).
11. Malandraki, C. and Daskin, M. S. (1992). Time dependent vehicle routing problems: formulations, properties and heuristic algorithms. *Transportation Science*, 26(3), 185-200.
13. Gendreau, M., Ghiani, G. and Guerriero, E. (2015). Time-dependent routing problems: a review. *Computers & Operations Research*, 64, 189-197. https://doi.org/10.1016/j.cor.2015.06.001 (accessed 18 October 2026).